
# Test-suite walkthrough

This notebook is a narrated version of the pytest suite in `refactor/tests/`.
Every test in that suite exists to pin down a specific piece of physics or
guard against a specific bug from the pre-refactor audit. Running each cell
executes the same assertions the test suite makes; the narrative around
each cell explains *what* piece of physics or code hygiene is being
verified.

Sections:

1. PBC primitives (`test_pbc.py`)
2. Circular-mean centre of mass (`test_com.py`)
3. Hydrogen-bond geometry (`test_hbond.py`)
4. Mean squared displacement (`test_msd.py`)
5. Radial distribution function (`test_rdf.py`)
6. Recombination detection (`test_recombination.py`)
7. Streaming vs eager parser parity (`test_streaming_parity.py`)
8. Water-box generator (`test_water_box.py`)
9. LAMMPS data parser (`test_lammps_data.py`)
10. Ion tracker (`test_ion_tracker.py`)
11. Species split (`test_species.py`)
12. End-to-end integration (`test_integration.py`)

Each cell prints a summary line so the whole notebook can be scanned for
`PASS` / `FAIL` quickly.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import mdwater
print("mdwater", mdwater.__version__)

def check(name, condition, extra=""):
    tag = "PASS" if condition else "FAIL"
    print(f"[{tag}] {name}" + (f"  ({extra})" if extra else ""))
    assert condition, name



## 1. PBC primitives

The `pbc` module is the single source of truth for minimum image, wrapping,
and unwrapping. The pre-refactor code had five duplicate implementations
and a `=- 1` typo in one of them (fixed in Section 2). These tests verify
the vector operations work correctly for both small and pathological input.


In [ ]:

from mdwater.pbc import (
    minimum_image, wrap_into_box, unwrap_trajectory, circular_mean, clip_for_ckdtree,
)

box = np.array([10.0, 10.0, 10.0])

# Small-vector case: no folding needed.
v = np.array([[1.0, 2.0, 3.0]])
check("minimum_image within half-box is identity", np.allclose(minimum_image(v, box), v))

# Folding past L/2.
check(
    "minimum_image folds |dx| > L/2",
    np.allclose(minimum_image(np.array([[6.0, -6.0, 0.0]]), box), [[-4.0, 4.0, 0.0]]),
)

# Multi-period wrap: pathological input.
box1 = np.array([1.0, 1.0, 1.0])
folded = minimum_image(np.array([[3.7, -2.3, 0.0]]), box1)
check("minimum_image handles multi-period offset", np.allclose(folded, [[-0.3, -0.3, 0.0]], atol=1e-12))

# wrap_into_box: negative and > L both handled.
w = wrap_into_box(np.array([[-1.0, 12.5, 5.0]]), box)
check("wrap_into_box normalises negatives and overshoots", np.allclose(w, [[9.0, 2.5, 5.0]]))

# clip_for_ckdtree strictly under upper bound.
clipped = clip_for_ckdtree(np.array([[5.0, 4.9999999999, 0.0]]), np.array([5.0, 5.0, 5.0]))
check("clip_for_ckdtree yields x < L", np.all(clipped < np.array([5.0, 5.0, 5.0])))



### 1b. `unwrap_trajectory` — the MSD fix

The pre-refactor MSD differenced *wrapped* coordinates. When a particle
crossed a box face, MSD spiked by L^2. `unwrap_trajectory` reconstructs
continuous coordinates by tracking integer image jumps.


In [ ]:

# Constant velocity 2 A / frame, wraps at frame 3.
positions = np.zeros((5, 1, 3))
positions[:, 0, 0] = [4.0, 6.0, 8.0, 0.0, 2.0]
unwrapped = unwrap_trajectory(positions, np.array([10.0, 10.0, 10.0]))
check(
    "unwrap_trajectory turns [4, 6, 8, 0, 2] into [4, 6, 8, 10, 12]",
    np.allclose(unwrapped[:, 0, 0], [4, 6, 8, 10, 12]),
)

# No-op on already-continuous input.
cts = np.arange(30).reshape(10, 1, 3).astype(np.float64)
check("unwrap_trajectory is a no-op on continuous input",
      np.allclose(unwrap_trajectory(cts, np.array([100., 100., 100.])), cts))



### 1c. `circular_mean`

Two atoms at scaled x = 0.95 and 0.05 straddle the boundary. Their CoM must
be near 0 or 1, NOT 0.5. This is the correct treatment used by the water
CoM in Section 2.


In [ ]:

coords = np.array([[0.95, 0.5, 0.5], [0.05, 0.5, 0.5]])
com = circular_mean(coords, np.array([1.0, 1.0]))
x = com[0]
check(
    "circular_mean across boundary lands near 0/1, not 0.5",
    min(x, 1.0 - x) < 1e-6,
    extra=f"x = {x:.6f}",
)

# Mass weighting: a heavier atom pulls the CoM toward it.
c_light = circular_mean(np.array([[0.1, 0, 0], [0.5, 0, 0]]), np.array([1.0, 1.0]))
c_heavy = circular_mean(np.array([[0.1, 0, 0], [0.5, 0, 0]]), np.array([1.0, 10.0]))
check("heavier partner pulls CoM toward itself", c_heavy[0] > c_light[0])



## 2. Centre of mass — the `=- 1` typo fix

The pre-refactor `get_com_dynamic` contained the string `temp[0] =- 1`,
which Python parses as `temp[0] = -1` (assignment to -1), not `-= 1`. Any
CoM slightly out of [0, 1) was silently clamped to ±1. The refactored
`com_dynamic` delegates to `circular_mean`, so any group size (OH-, H2O,
H3O+) is handled without the bug.


In [ ]:

from mdwater.geometry.com import com_water, com_dynamic

# Symmetric H around O -> CoM at O.
com = com_water(
    np.array([[[0.55, 0.5, 0.5], [0.45, 0.5, 0.5]]]),
    np.array([[0.5, 0.5, 0.5]]),
)
check("com_water on interior molecule matches O position", np.allclose(com[0], [0.5, 0.5, 0.5], atol=1e-6))

# Straddling the boundary -> CoM near 0.
com = com_water(
    np.array([[[0.99, 0.5, 0.5], [0.03, 0.5, 0.5]]]),
    np.array([[0.01, 0.5, 0.5]]),
)
x = com[0, 0]
check("com_water across boundary lands near 0/1", min(x, 1.0 - x) < 0.05, extra=f"x = {x:.4f}")

# All three molecule sizes.
H = np.array([
    [0.10, 0.10, 0.10],
    [0.20, 0.10, 0.10],
    [0.30, 0.10, 0.10],
    [0.40, 0.10, 0.10],
])
O = np.array([[0.15, 0.10, 0.10], [0.35, 0.10, 0.10]])
molecules = [[0, 0], [1, 2, 1], [1, 2, 3, 1]]  # OH-, H2O, H3O+
com = com_dynamic(molecules, H, O)
check(
    "com_dynamic handles OH-, H2O, H3O+ and stays in [0, 1)",
    np.all(com >= 0.0) and np.all(com < 1.0),
)

# Regression for the `=- 1` bug: CoM near boundary should NOT equal ±1.
com = com_dynamic([[0, 0]], np.array([[0.98, 0.98, 0.98]]), np.array([[0.02, 0.02, 0.02]]))
check(
    "regression: CoM never clipped to +/- 1 (legacy bug)",
    not np.any(com == 1.0) and not np.any(com == -1.0),
)



## 3. Hydrogen bonds — angle convention

The pre-refactor `check_hbond` built `r_hd = OD - H` (correct) and
`r_ha = H - OA` (wrong sign). For a linear D-H...A geometry these vectors
were *parallel*, so `arccos` returned ~0 degrees, and the `theta >= 150`
threshold silently rejected every linear H-bond it was supposed to accept.


In [ ]:

from mdwater.observables.hbond import find_hydrogen_bonds, build_hbond_wire, HBond
from mdwater.config import HBondConfig

box = np.array([20.0, 20.0, 20.0])

# Linear geometry: D at (0,0,3), H at (0,0,2), A at (0,0,0).
oxy = np.array([[0., 0., 0.], [0., 0., 3.]])
hyd = np.array([[0., 0., 2.]])
h_to_o = np.array([1])
bonds = find_hydrogen_bonds(hyd, oxy, h_to_o, box, HBondConfig())
check("linear D-H...A accepted at theta = 180 deg",
      len(bonds) == 1 and abs(bonds[0].dha_angle_deg - 180.0) < 1e-3,
      extra=f"theta = {bonds[0].dha_angle_deg:.2f}" if bonds else "no bond")

# Perpendicular geometry: rejected.
oxy = np.array([[0., 0., 0.], [3., 0., 0.]])
hyd = np.array([[0., 0., 1.]])
h_to_o = np.array([1])
bonds = find_hydrogen_bonds(hyd, oxy, h_to_o, box, HBondConfig())
check("perpendicular geometry rejected", bonds == [])

# Far pair beyond O-O cutoff.
oxy = np.array([[0., 0., 0.], [0., 0., 5.0]])
hyd = np.array([[0., 0., 4.0]])
h_to_o = np.array([1])
bonds = find_hydrogen_bonds(hyd, oxy, h_to_o, box, HBondConfig())
check("far pair (OO > 3.5) rejected", bonds == [])

# H-bond across a periodic boundary.
box_small = np.array([10., 10., 10.])
oxy = np.array([[0.5, 0., 0.], [9.5, 0., 0.]])
hyd = np.array([[9.9, 0., 0.]])
h_to_o = np.array([1])
bonds = find_hydrogen_bonds(hyd, oxy, h_to_o, box_small, HBondConfig())
check("H-bond across PBC accepted, OO = 1 A",
      len(bonds) == 1 and abs(bonds[0].oo_distance - 1.0) < 1e-6)

# H-bond wire BFS.
graph = [HBond(0, 0, 1, 2.8, 170), HBond(1, 1, 2, 2.8, 170), HBond(2, 2, 3, 2.8, 170)]
paths = build_hbond_wire(0, graph, max_depth=3)
check("BFS enumerates the 3-hop wire 0->1->2->3", [0, 1, 2, 3] in paths)



## 4. MSD — PBC unwrap + Einstein-relation fit

Two categories: analytic verification (ballistic + Brownian) and pathology
regressions (particle crossing a face).


In [ ]:

from mdwater.observables import compute_msd, translational_diffusion
from mdwater.config import MSDConfig

box = np.array([10., 10., 10.])

# Zero drift -> zero MSD.
positions = np.tile([[[1.0, 2.0, 3.0]]], (10, 5, 1))
r = compute_msd(positions, box, MSDConfig(timestep_ps=1.0))
check("zero drift -> zero MSD", np.allclose(r.msd, 0.0))

# Constant velocity crossing a face: MSD must be smooth quadratic.
T = 20
v = np.array([1.5, 0., 0.])
x0 = np.array([[0.5, 0.5, 0.5]])
positions = np.zeros((T, 1, 3))
for t in range(T):
    positions[t, 0] = np.mod(x0 + v * t, box)
r = compute_msd(positions, box, MSDConfig(timestep_ps=1.0))
expected = (v[0] * np.arange(T)) ** 2
check("wrapped ballistic particle -> quadratic MSD (v t)^2",
      np.allclose(r.msd, expected, atol=1e-8))

# Free 3D Brownian ensemble: D should be 0.5 for unit-variance steps in 3D.
rng = np.random.default_rng(0)
T, N = 300, 500
steps = rng.standard_normal((T, N, 3))
positions = np.mod(np.cumsum(steps, axis=0), 100.0)
r = compute_msd(positions, np.array([100., 100., 100.]), MSDConfig(timestep_ps=1.0))
D = translational_diffusion(r, MSDConfig(timestep_ps=1.0, fit_range=(0.2, 0.8)))
check("Brownian ensemble D within [0.4, 0.6]", 0.4 < D < 0.6, extra=f"D = {D:.4f}")



## 5. RDF — normalization, cutoff, hard-core

The important behaviours:

- Uniform gas: g(r) -> 1 in the tail.
- Hard-core rejection: g(r) = 0 below the minimum distance.
- Cutoff request > L/2: clamped to L/2 (safe for NPT).


In [ ]:

from mdwater.observables import compute_rdf
from mdwater.config import RDFConfig

# Uniform gas -> g(r) tail approaches 1.
rng = np.random.default_rng(0)
L = 20.0
positions = rng.uniform(0.0, L, size=(5, 800, 3))
box = np.tile([L, L, L], (5, 1))
r = compute_rdf(positions, positions, box, "OO", RDFConfig(n_bins=40, r_min_angstrom=0.5))
tail = r.gr[r.r > 5.0].mean()
check("uniform gas g(r) tail approx 1", 0.85 < tail < 1.15, extra=f"tail mean = {tail:.3f}")

# Requested cutoff > L/2 is clamped.
L = 6.0
positions = rng.uniform(0.0, L, size=(2, 200, 3))
box = np.tile([L, L, L], (2, 1))
r = compute_rdf(positions, positions, box, "OO",
                RDFConfig(n_bins=20, r_min_angstrom=0.1, r_max_angstrom=10.0))
check("RDF cutoff clamped to L/2", r.r.max() <= L / 2)



## 6. Recombination detection

The pre-refactor detector did binary search assuming monotone transition —
brittle in the presence of Grotthuss shuttling. The refactor uses a
linear scan with a configurable minimum ion-free dwell window.


In [ ]:

from mdwater.ions.tracker import IonFrame, IonTrajectory
from mdwater.ions.recombination import detect_recombination
from mdwater.config import RecombinationConfig

def flags_to_traj(flags):
    frames = []
    for f in flags:
        if f:
            frames.append(IonFrame(np.array([0], dtype=np.int64),
                                    np.array([1], dtype=np.int64),
                                    np.array([1, 3], dtype=np.int64)))
        else:
            frames.append(IonFrame(np.array([], dtype=np.int64),
                                    np.array([], dtype=np.int64),
                                    np.array([2, 2], dtype=np.int64)))
    return IonTrajectory(per_frame=frames)

cfg = RecombinationConfig(min_dwell_frames=5)

# Case A: transient 2-frame ion-free window followed by more ions.
res = detect_recombination(flags_to_traj([True]*5 + [False]*2 + [True]*3 + [False]*3), cfg)
check("transient 2-frame ion-free window NOT accepted", res.recombined is False)

# Case B: sustained 10-frame ion-free window.
res = detect_recombination(flags_to_traj([True]*5 + [False]*10), cfg)
check("sustained ion-free window accepted at t = 5",
      res.recombined is True and res.frame == 5)

# Case C: pure water throughout.
res = detect_recombination(flags_to_traj([False]*20), RecombinationConfig(min_dwell_frames=3))
check("never-ionised trajectory recombines at t = 0",
      res.recombined is True and res.frame == 0)

# Case D: never recombines.
res = detect_recombination(flags_to_traj([True]*20), RecombinationConfig(min_dwell_frames=3))
check("never-recombining trajectory reports frame = n_snapshots",
      res.recombined is False and res.frame == 20)



## 7. Streaming vs eager parser parity

The pre-refactor code had two separate LAMMPS trajectory parsers that
drifted (one had an off-by-one, the other had `np.fromstring` deprecation
warnings). In the refactor they share every header-parsing routine, and
this test round-trips a synthetic file through both paths.


In [ ]:

from pathlib import Path
import tempfile
from mdwater.io.writers import write_lammpstrj
from mdwater.io.lammpstrj import read_lammpstrj
from mdwater.io.lammpstrj_stream import stream_lammpstrj_to_hdf5
from mdwater.io.hdf5_backend import load_hdf5_trajectory

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    T, N = 4, 6
    rng = np.random.default_rng(0)
    atoms = np.zeros((T, N, 5))
    box_dim = np.zeros((T, 3, 2)); box_dim[:, :, 1] = 10.0
    for t in range(T):
        for i in range(N):
            atoms[t, i, 0] = i + 1
            atoms[t, i, 1] = 2 if i < 2 else 1
            atoms[t, i, 2:5] = rng.uniform(0.0, 10.0, size=3)
    src = tmp / "toy.lammpstrj"
    write_lammpstrj(src, atoms, box_dim, scaled=False)

    eager_atoms, eager_box, _ = read_lammpstrj(src)
    stream_lammpstrj_to_hdf5(src, tmp / "toy.h5")
    with load_hdf5_trajectory(tmp / "toy.h5", mode="full") as hdf:
        streamed_atoms = np.asarray(hdf.atoms)
        streamed_box = np.asarray(hdf.box)

check("eager and streaming parsers agree on atoms",
      eager_atoms.shape == streamed_atoms.shape and np.allclose(eager_atoms, streamed_atoms))
check("eager and streaming parsers agree on box",
      np.allclose(eager_box, streamed_box))



## 8. Water-box generator physics

Regressions on: minimum O-O respected, correct stoichiometry, and
determinism given a seed.


In [ ]:

from mdwater.water_box import WaterBoxSpec, generate_water_box
from mdwater.geometry.neighbors import build_kdtree

spec = WaterBoxSpec(n_molecules=32, number_density=0.0334, min_OO=2.5, seed=42)
box_obj = generate_water_box(spec)

check("generator returns 2 H per O", box_obj.H_positions.shape[0] == 2 * box_obj.O_positions.shape[0])
check("generator returns the requested number of molecules", box_obj.O_positions.shape[0] == 32)

# Minimum O-O respected within a small tolerance (soft repulsion isn't hard-sphere).
tree = build_kdtree(box_obj.O_positions, box_obj.box)
d, _ = tree.query(clip_for_ckdtree(box_obj.O_positions, box_obj.box), k=2)
check("nearest O-O >= min_OO - epsilon", d[:, 1].min() >= 2.4, extra=f"min nn = {d[:, 1].min():.3f}")

# Deterministic on repeated calls with the same seed.
a = generate_water_box(WaterBoxSpec(n_molecules=8, number_density=0.03, seed=7))
b = generate_water_box(WaterBoxSpec(n_molecules=8, number_density=0.03, seed=7))
check("seeded RNG makes the generator deterministic",
      np.allclose(a.O_positions, b.O_positions) and np.allclose(a.H_positions, b.H_positions))



## 9. LAMMPS data parser — n_atoms is read from the file

The pre-refactor parser had `n_atoms = 1824` hardcoded in the middle of the
loop, overriding whatever the file declared. The refactored parser accepts
any atom count, verified across 12, 300, and 1500.


In [ ]:

from mdwater.io.lammps_data import read_lammps_data

def write_data_file(path, n_atoms, box_L=10.0):
    with path.open("w") as f:
        f.write("# fixture\n\n")
        f.write(f"{n_atoms} atoms\n")
        f.write("2 atom types\n\n")
        f.write(f"0.0 {box_L} xlo xhi\n")
        f.write(f"0.0 {box_L} ylo yhi\n")
        f.write(f"0.0 {box_L} zlo zhi\n\n")
        f.write("Masses\n\n1 1.00784\n2 15.999\n\n")
        f.write("Atoms # atomic\n\n")
        for i in range(1, n_atoms + 1):
            typ = 2 if i % 3 == 0 else 1
            f.write(f"{i} {typ} {i*0.1:.3f} {i*0.1:.3f} {i*0.1:.3f}\n")

import tempfile
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for n in (12, 300, 1500):
        p = tmp / f"n{n}.data"
        write_data_file(p, n)
        result = read_lammps_data(p)
        check(f"n_atoms parsed correctly for n = {n}",
              result.n_atoms == n and result.atoms.shape == (1, n, 5))



## 10. Ion tracker

Pure water has no ions; explicit H3O+ and OH- geometries are correctly
tagged; and multi-ion frames survive (the pre-refactor code kept only the
first of each).


In [ ]:

from mdwater.ions.tracker import identify_ions

# Pure water: 4 H2O, each O has 2 H.
L = 10.0; box = np.array([L, L, L])
oxy = np.array([[1., 1., 1.], [4., 1., 1.], [7., 1., 1.], [1., 5., 1.]])
hyd = np.stack([o + np.array([sx, 0., 0.]) for o in oxy for sx in (0.3, -0.3)])
frame = identify_ions(hyd, oxy, box)
check("pure water: no ions detected",
      frame.oh_indices.size == 0 and frame.h3o_indices.size == 0)

# 1 H3O+ + 1 OH-.
oxy = np.array([[1., 1., 1.], [8., 1., 1.]])
hyd = np.array([[1.3, 1., 1.], [0.7, 1., 1.], [1., 1.3, 1.], [8.3, 1., 1.]])
frame = identify_ions(hyd, oxy, np.array([20., 20., 20.]))
check("single H3O+ and OH- correctly identified",
      list(frame.h3o_indices) == [0] and list(frame.oh_indices) == [1])

# Two H3O+ + two OH-.
oxy = np.array([[1., 1., 1.], [8., 1., 1.], [15., 1., 1.], [22., 1., 1.]])
hyd = []
for idx in (0, 2):
    hyd += [oxy[idx] + [0.3, 0, 0], oxy[idx] + [-0.3, 0, 0], oxy[idx] + [0, 0.3, 0]]
for idx in (1, 3):
    hyd += [oxy[idx] + [0.3, 0, 0]]
hyd = np.stack(hyd)
frame = identify_ions(hyd, oxy, np.array([30., 30., 30.]))
check("multi-ion frame preserves all four ion indices",
      sorted(frame.h3o_indices.tolist()) == [0, 2] and sorted(frame.oh_indices.tolist()) == [1, 3])



## 11. Species split — no magic ints, index stability enforced


In [ ]:

from mdwater.species import split_species
from mdwater.config import AtomTypes
from mdwater.errors import InconsistentTrajectoryError

# Default (H=1, O=2).
T, N = 3, 12
traj = np.zeros((T, N, 5))
for t in range(T):
    traj[t, :, 1] = [1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 2]
split = split_species(traj)
check("split_species with default AtomTypes: 8 H and 4 O",
      split.hydrogen.shape == (T, 8, 5) and split.oxygen.shape == (T, 4, 5))

# Custom types.
traj = np.zeros((1, 4, 5))
traj[0, :, 1] = [5, 5, 6, 6]
split = split_species(traj, atom_types=AtomTypes(hydrogen=5, oxygen=6))
check("split_species accepts custom atom-type mapping",
      split.hydrogen.shape == (1, 2, 5) and split.oxygen.shape == (1, 2, 5))

# Renumbering across frames -> error.
traj = np.zeros((3, 4, 5))
traj[0, :, 1] = [1, 1, 2, 2]
traj[1, :, 1] = [1, 1, 2, 2]
traj[2, :, 1] = [2, 1, 1, 2]   # swapped
try:
    split_species(traj)
    check("split_species detects index drift across frames", False)
except InconsistentTrajectoryError:
    check("split_species detects index drift across frames", True)



## 12. Real trajectory: `n_608` HDNN run (300 K, 608 waters, 301 frames)

Every check above uses synthetic data — good for pinning behavior but not
a substitute for real physics. This section loads
`Z:\cluster_runs\n_608\expanded_run\trjwater.lammpstrj` (1824 atoms,
32.5 A cubic box, scaled coordinates in the file, 301 frames dumped every
few fs) and verifies each observable produces a physically sensible
result.

The path is hard-coded to the shared Z: drive. If the file is not
available on your machine, the cell prints a message and skips gracefully.


In [ ]:

import time
from mdwater import Trajectory, RDFConfig, HBondConfig, MSDConfig
from mdwater.observables import (
    compute_rdf, compute_ion_rdf, find_hydrogen_bonds,
    compute_msd, translational_diffusion, ion_pair_distance,
)
from mdwater.ions import track_ions, detect_recombination

REAL_TRAJ = Path(r"Z:\cluster_runs\n_608\expanded_run\trjwater.lammpstrj")
HAS_REAL_DATA = REAL_TRAJ.exists()
print("real trajectory present:", HAS_REAL_DATA)
if not HAS_REAL_DATA:
    print("SKIPPING real-data cells — file not available on this machine.")



### 12a. Load and sanity-check the trajectory

Row order in LAMMPS `dump custom` output is not guaranteed to be stable
across frames (many-body integrators reorder atoms). `Trajectory.from_lammpstrj`
sorts each frame by atom id so row-indexed access always refers to the
same physical atom.


In [ ]:

if HAS_REAL_DATA:
    t0 = time.time()
    trj_real = Trajectory.from_lammpstrj(REAL_TRAJ)
    load_t = time.time() - t0
    print(f"load time     : {load_t:.2f} s")
    print(f"n_snapshots   : {trj_real.n_snapshots}")
    print(f"n_atoms       : {trj_real.n_atoms}")
    print(f"box (A)       : {trj_real.box_size[0]}")
    n_H = trj_real.species.hydrogen.shape[1]
    n_O = trj_real.species.oxygen.shape[1]
    print(f"H : O         : {n_H} : {n_O}   (ratio = {n_H / n_O:.3f})")

    check("Water stoichiometry preserved: exactly 2 H per O",
          n_H == 2 * n_O)
    check("Box is orthogonal (~32.49 A cubic)",
          np.allclose(trj_real.box_size[0], trj_real.box_size[0, 0]))
    check("Species indexing stable after id sort (loading succeeded)",
          trj_real.species.hydrogen.shape == (trj_real.n_snapshots, n_H, 5))



### 12b. O-O RDF against experiment

Ambient liquid water has a characteristic O-O first peak at r ≈ 2.75-2.85 A
(experimental neutron / X-ray). Classical NN-potential simulations at 300 K
usually reproduce this to within ~0.1 A, with a peak height in the 2.0-3.0
range. The RDF tail must approach 1.


In [ ]:

if HAS_REAL_DATA:
    rdf_oo = trj_real.rdf("OO", RDFConfig(n_bins=200, r_min_angstrom=0.5))
    peak_idx = rdf_oo.gr.argmax()
    peak_r = rdf_oo.r[peak_idx]
    peak_g = rdf_oo.gr[peak_idx]
    # Second peak: search after the first minimum.
    first_min_idx = peak_idx + np.argmin(rdf_oo.gr[peak_idx:peak_idx + 50])
    second_peak_idx = first_min_idx + np.argmax(rdf_oo.gr[first_min_idx:first_min_idx + 60])
    second_peak_r = rdf_oo.r[second_peak_idx]

    tail = rdf_oo.gr[rdf_oo.r > 10.0].mean()
    print(f"first peak    : r = {peak_r:.2f} A, g = {peak_g:.2f}")
    print(f"second peak   : r = {second_peak_r:.2f} A")
    print(f"tail mean     : {tail:.3f}   (expect ~1.0)")

    check("O-O first peak in physical range 2.5-3.5 A",
          2.5 <= peak_r <= 3.5, extra=f"r = {peak_r:.2f}")
    check("O-O first peak height is realistic (1.8-4.0)",
          1.8 <= peak_g <= 4.0, extra=f"g_peak = {peak_g:.2f}")
    check("O-O second peak between 4-6 A",
          4.0 <= second_peak_r <= 6.0, extra=f"r_2 = {second_peak_r:.2f}")
    check("g_OO(r) tail approaches 1", 0.8 <= tail <= 1.2)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(rdf_oo.r, rdf_oo.gr, "-", label="g_OO(r)")
    ax.axhline(1.0, color="k", linestyle=":")
    ax.axvline(2.8, color="g", linestyle="--", alpha=0.5, label="exp. ~2.8 A")
    ax.set(xlabel="r (A)", ylabel="g(r)",
           title=f"O-O RDF, {trj_real.n_snapshots} frames of 608-water HDNN run")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()



### 12c. O-H and H-H RDFs

- g_OH(r) has a sharp intra-molecular peak at ~1.0 A (rigid O-H bond) and
  the first inter-molecular H-bond peak near 1.85 A.
- g_HH(r) has an intra-molecular peak near 1.55 A (H-O-H opening).


In [ ]:

if HAS_REAL_DATA:
    rdf_oh = trj_real.rdf("OH", RDFConfig(n_bins=200, r_min_angstrom=0.5))
    rdf_hh = trj_real.rdf("HH", RDFConfig(n_bins=200, r_min_angstrom=0.5))

    # First OH peak (intramolecular O-H).
    oh_peak_r = rdf_oh.r[rdf_oh.gr.argmax()]
    # First HH peak.
    hh_peak_r = rdf_hh.r[rdf_hh.gr.argmax()]
    print(f"g_OH first peak: r = {oh_peak_r:.2f} A  (expect ~1.0 A intramolecular)")
    print(f"g_HH first peak: r = {hh_peak_r:.2f} A  (expect ~1.55 A intramolecular)")

    check("g_OH first (intramolecular) peak near r_OH = 1.0 A",
          0.8 <= oh_peak_r <= 1.2, extra=f"r = {oh_peak_r:.2f}")
    check("g_HH first (intramolecular) peak near 1.4-1.7 A",
          1.3 <= hh_peak_r <= 1.8, extra=f"r = {hh_peak_r:.2f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    ax1.plot(rdf_oh.r, rdf_oh.gr); ax1.set(xlabel="r (A)", ylabel="g_OH(r)"); ax1.grid(alpha=0.3)
    ax2.plot(rdf_hh.r, rdf_hh.gr); ax2.set(xlabel="r (A)", ylabel="g_HH(r)"); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()



### 12d. Hydrogen-bond count per water

The Luzar-Chandler criterion (O-O < 3.5 A, DHA angle > 150 deg) is applied
per frame. `find_hydrogen_bonds` returns each bond exactly once (from the
donor side), so a "participation" count -- the more common per-molecule
figure in the literature -- is 2 * (bonds / N_water). Ambient water at
300 K sits around 3.5 participations, i.e. ~1.75 unique bonds per water.

The first frame of this trajectory is often an unequilibrated seed; we
sample from `n // 5` onward for the physical check.


In [ ]:

if HAS_REAL_DATA:
    n_water = trj_real.species.oxygen.shape[1]
    hbond_config = HBondConfig()  # O-O <= 3.5 A, angle >= 150 deg
    counts = []
    sample_frames = np.linspace(trj_real.n_snapshots // 5,
                                trj_real.n_snapshots - 1, 20, dtype=int)
    for t in sample_frames:
        bonds = trj_real.hydrogen_bonds(int(t), hbond_config)
        counts.append(len(bonds))
    counts = np.asarray(counts)
    unique_per_water = counts / n_water
    participation_per_water = 2.0 * unique_per_water
    print(f"unique H-bonds / water        : mean = {unique_per_water.mean():.2f}, std = {unique_per_water.std():.2f}")
    print(f"participation H-bonds / water : mean = {participation_per_water.mean():.2f}")
    print(f"(reference: ~1.75 unique / ~3.5 participation for ambient water)")

    check("participation H-bonds per water in physical range (2.0 - 4.5)",
          2.0 <= participation_per_water.mean() <= 4.5,
          extra=f"<n_HB_part> = {participation_per_water.mean():.2f}")
    check("H-bond count is fairly stable across the sampled window",
          counts.std() / max(counts.mean(), 1) < 0.15)

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(sample_frames, participation_per_water, "o-", label="2 * bonds / N_water")
    ax.axhspan(3.3, 3.8, color="g", alpha=0.15, label="expected ~3.5")
    ax.set(xlabel="frame", ylabel="participation H-bonds per water",
           title="Luzar-Chandler H-bond density")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()



### 12e. Oxygen MSD and translational diffusion

The refactored MSD unwraps PBC and works in Angstrom. This particular
trajectory (`n_608/expanded_run`) shows a mean per-oxygen displacement of
~8 A over its full length -- the exact D depends on the (unknown to us)
frame-to-frame timestep, so we only check the *mechanics*:

- The MSD grows across the trajectory (particles diffuse forward).
- The Einstein-relation fit returns a positive, finite D.
- Absolute physical calibration would require knowing the dumped `dt`.


In [ ]:

if HAS_REAL_DATA:
    # We do not know the exact dt between dumped frames from the file;
    # use 20 fs (0.02 ps) as a plausible HDNN production spacing so the
    # x-axis is in ps for plotting. D scales as 1/dt so the *unit* changes
    # but the mechanics test is unaffected.
    dt_ps = 0.02
    result = trj_real.msd_oxygen(MSDConfig(timestep_ps=dt_ps))
    D = trj_real.translational_diffusion(
        result, MSDConfig(timestep_ps=dt_ps, fit_range=(0.3, 0.9)),
    )
    print(f"n_lags        : {result.msd.size}")
    print(f"MSD tail (A^2): {result.msd[-1]:.2f}")
    print(f"D (assumed 20 fs frame spacing): {D:.4f} A^2/ps")
    print("(Absolute D calibration requires knowing the true dt.)")

    late = result.msd[len(result.msd) // 4:]
    check("MSD grows on average across the diffusive regime",
          late[-1] > late[0],
          extra=f"MSD[T/4] = {late[0]:.2f}, MSD[T] = {late[-1]:.2f}")
    check("Diffusion coefficient is positive and finite",
          D > 0 and np.isfinite(D), extra=f"D = {D:.4f}")

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(result.t, result.msd, label="oxygen MSD")
    ax.plot(result.t, 6.0 * D * result.t, "k--",
            label=f"6 D t, D = {D:.3f} A^2/ps")
    ax.set(xlabel="t (ps, assumed dt=20 fs)", ylabel="MSD (A^2)",
           title="Oxygen mean squared displacement")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()



### 12f. Ion tracking on a pure-water trajectory

This trajectory is pure water — no ions were seeded. The ion tracker
should therefore detect *no* OH- and *no* H3O+ in any frame, and the
recombination detector should trivially report the trajectory as
"already recombined" at frame 0.


In [ ]:

if HAS_REAL_DATA:
    ion_traj = trj_real.ion_trajectory()
    total_h3o = sum(f.h3o_indices.size for f in ion_traj.per_frame)
    total_oh  = sum(f.oh_indices.size  for f in ion_traj.per_frame)
    print(f"total H3O+ instances across all {trj_real.n_snapshots} frames: {total_h3o}")
    print(f"total OH-  instances across all {trj_real.n_snapshots} frames: {total_oh}")

    check("pure-water trajectory has zero H3O+ frames", total_h3o == 0)
    check("pure-water trajectory has zero OH-  frames", total_oh == 0)

    result = trj_real.recombination(RecombinationConfig(min_dwell_frames=10))
    print(f"recombination result: recombined={result.recombined}, frame={result.frame}")
    check("recombination detector accepts pure water at frame 0",
          result.recombined is True and result.frame == 0)



### 12g. Streaming HDF5 vs eager parser: parity on real data

The eager parser and the streaming HDF5 converter must produce identical
`(atoms, box)` arrays for the full real trajectory too, not just for toy
inputs. This is the check that guards against silent divergence of the
two paths.


In [ ]:

if HAS_REAL_DATA:
    from mdwater.io.lammpstrj_stream import stream_lammpstrj_to_hdf5
    from mdwater.io.hdf5_backend import load_hdf5_trajectory
    from mdwater.io.lammpstrj import read_lammpstrj

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        h5 = tmp / "trjwater.h5"
        t0 = time.time()
        stream_lammpstrj_to_hdf5(REAL_TRAJ, h5, overwrite=True)
        stream_t = time.time() - t0

        with load_hdf5_trajectory(h5, mode="full") as hdf:
            streamed_atoms = np.asarray(hdf.atoms)
            streamed_box   = np.asarray(hdf.box)

    # Streaming preserves the file's raw representation. Compare against
    # an eager parse with the same setting so we're testing parser parity,
    # not coordinate-conversion parity.
    eager_atoms_raw, eager_box_raw, _ = read_lammpstrj(REAL_TRAJ, scale="as_is")

    print(f"streaming conversion time: {stream_t:.2f} s")
    print(f"shapes match             : {eager_atoms_raw.shape == streamed_atoms.shape}")

    check("streaming and eager atoms arrays are bitwise-equal (up to fp)",
          np.allclose(eager_atoms_raw, streamed_atoms))
    check("streaming and eager box_dim arrays are bitwise-equal",
          np.allclose(eager_box_raw, streamed_box))



## 13. End-to-end integration (synthetic)

The final battery generates a water box, writes it to disk, loads it back
through the `Trajectory` facade, and runs an RDF plus recombination check.
Complementary to the real-data section: catches breakage in the write /
read / observable pipeline.


In [ ]:

from mdwater import Trajectory, RDFConfig
from mdwater.water_box import WaterBoxSpec, generate_water_box, write_lammps_data
from mdwater.io.writers import write_lammpstrj

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    spec = WaterBoxSpec(n_molecules=64, number_density=0.028, min_OO=2.4, seed=1)
    box_obj = generate_water_box(spec)

    # Round-trip through .data.
    dpath = tmp / "water.data"
    write_lammps_data(box_obj, dpath)
    trj = Trajectory.from_lammps_data(dpath)
    check("data-file round-trip preserves atom count (no hardcoded n_atoms)",
          trj.n_atoms == 64 * 3)

    # Fake 4-frame trajectory around the water box, round-trip through .lammpstrj.
    rng = np.random.default_rng(0)
    T = 4
    n_o = box_obj.O_positions.shape[0]
    n_h = box_obj.H_positions.shape[0]
    atoms = np.zeros((T, n_o + n_h, 5))
    box_dim = np.zeros((T, 3, 2))
    box_dim[:, :, 1] = box_obj.box
    for t in range(T):
        atoms[t, :n_o, 0] = np.arange(1, n_o + 1)
        atoms[t, :n_o, 1] = 2
        atoms[t, :n_o, 2:5] = np.mod(box_obj.O_positions + 0.02 * rng.standard_normal((n_o, 3)), box_obj.box)
        atoms[t, n_o:, 0] = np.arange(n_o + 1, n_o + n_h + 1)
        atoms[t, n_o:, 1] = 1
        atoms[t, n_o:, 2:5] = np.mod(box_obj.H_positions + 0.02 * rng.standard_normal((n_h, 3)), box_obj.box)
    tpath = tmp / "traj.lammpstrj"
    write_lammpstrj(tpath, atoms, box_dim, scaled=False)
    trj = Trajectory.from_lammpstrj(tpath)
    check("lammpstrj round-trip: correct frame count", trj.n_snapshots == 4)

    # RDF peak between 2 and 5 A (first hydration shell region).
    r = trj.rdf("OO", RDFConfig(n_bins=30, r_min_angstrom=0.5))
    peak = r.r[np.argmax(r.gr)]
    check("RDF has a first-shell-like peak in [2, 5] A", 2.0 <= peak <= 5.0,
          extra=f"peak at {peak:.2f} A")

    # Pure water throughout -> recombination detected at frame 0.
    # Trajectory is only 4 frames long, so lower the dwell requirement.
    result = trj.recombination(RecombinationConfig(min_dwell_frames=2))
    check("pure-water trajectory reports recombined = True", result.recombined is True)

print("All integration checks completed.")
